# Week 08 · 중간 실기: 성적 분석 미니 프로젝트

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MNU-AI-Programming/AI-Assisted-Python-Programming/blob/main/weeks/week08/assignment.ipynb)

## 실기 안내

- **범위**: 1~7주차 (문자열·제어문·자료구조·함수·파일·예외·클래스). pandas는 사용하지 않습니다.
- **데이터**: `students.csv` – 80명의 가상 성적 데이터
- **AI 사용**: 교수자가 공지한 범위를 따릅니다. 사용했다면 **기술 메모에 반드시 기록**합니다.
- **제출**: 파일 이름 `midterm_학번_이름.ipynb`, **런타임 → 다시 시작 및 모두 실행** 후 제출

| 문제 | 내용 | 배점 |
|---|---|---:|
| 1 | 데이터 읽기와 검증 | 20 |
| 2 | 성적 계산과 학과별 요약 | 25 |
| 3 | `GradeBook` 클래스 | 25 |
| 4 | 보고서 파일 만들기 | 20 |
| 5 | 기술 메모 | 10 |
| | **합계** | **100** |

### 성적 규정 (모든 문제 공통)

| 항목 | 규정 |
|---|---|
| 총점 | 중간 30% + 기말 40% + 과제 30%, **소수 첫째 자리 반올림** (`round(x, 1)`) |
| 과제 미제출 | 빈 칸이면 `None`으로 읽고, 총점 계산에서는 **0점** |
| 등급 | 90 이상 A, 80 이상 B, 70 이상 C, 60 이상 D, 그 외 F |
| 출석 | 16회 중 **12회 미만이면 총점과 관계없이 F** |

> 자동 확인은 최소 기준입니다. 채점자는 코드를 직접 읽고, 결과를 하드코딩한 코드는 점수를 받지 못합니다.

## 준비: 데이터와 자동 확인 도우미

In [ ]:
import csv
import tempfile
import urllib.request
from pathlib import Path

DATA_URL = "https://raw.githubusercontent.com/MNU-AI-Programming/AI-Assisted-Python-Programming/main/datasets/"

def get_data(name):
    for folder in [Path("../../datasets"), Path("datasets"), Path(".")]:
        if (folder / name).exists():
            return str(folder / name)
    urllib.request.urlretrieve(DATA_URL + name, name)
    return name

STUDENTS = get_data("students.csv")
TMP = Path(tempfile.mkdtemp())


def check(desc, test):
    """자동 확인 도우미: test는 True/False를 돌려주는 함수(lambda)입니다."""
    try:
        ok = test()
    except NotImplementedError:
        print(f"⏳ 미작성 - {desc}")
    except Exception as e:
        print(f"❌ 오류 - {desc} ({type(e).__name__}: {e})")
    else:
        print(f"{'✅ 통과' if ok else '❌ 실패'} - {desc}")

---
## 문제 1. 데이터 읽기와 검증 (20점)

`load_students(path)`는 CSV를 읽어 **`(레코드 리스트, 건너뛴 행 수)`** 를 돌려줍니다.

각 레코드는 다음 딕셔너리입니다.

```python
{"student_id": "20261000", "name": "황서윤", "department": "간호학과", "year": 1,
 "midterm": 100, "final": 100, "assignment": 100, "attendance": 14}
```

- `year`, `midterm`, `final`, `attendance`는 `int`, `assignment`는 `int` 또는 (빈 칸이면) `None`
- 앞뒤 공백은 제거합니다.
- 다음 행은 **건너뛰고 개수를 셉니다**: 필수 값(학번·이름·학년·중간·기말·출석)이 비었거나 정수가 아님, 점수(중간·기말·과제)가 0~100 밖, 출석이 0~16 밖, 학년이 1~4 밖
- `except:`처럼 모든 오류를 잡는 코드는 쓰지 않습니다.

In [ ]:
def load_students(path):
    """(레코드 리스트, 건너뛴 행 수)를 돌려줍니다."""
    # TODO: 여기에 코드를 작성하세요
    raise NotImplementedError("코드를 작성하세요")

In [ ]:
check("students.csv 80명, 건너뜀 0", lambda: (len(load_students(STUDENTS)[0]), load_students(STUDENTS)[1]) == (80, 0))
check("숫자 열은 int", lambda: all(type(r["midterm"]) is int and type(r["attendance"]) is int for r in load_students(STUDENTS)[0]))
check("과제 빈 칸 → None (3명)", lambda: sum(r["assignment"] is None for r in load_students(STUDENTS)[0]) == 3)
check("첫 레코드", lambda: load_students(STUDENTS)[0][0] == {"student_id": "20261000", "name": "황서윤", "department": "간호학과",
      "year": 1, "midterm": 100, "final": 100, "assignment": 100, "attendance": 14})

bad = TMP / "bad.csv"
bad.write_text("\n".join([
    "student_id,name,department,year,midterm,final,assignment,attendance",
    "1,가,X학과,1,80,90,,16",           # 정상 (과제 없음)
    "2,,X학과,1,80,90,70,16",            # 이름 없음
    "3,다,X학과,5,80,90,70,16",          # 학년 범위
    "4,라,X학과,2,abc,90,70,16",         # 숫자 아님
    "5,마,X학과,2,80,101,70,16",         # 점수 범위
    "6,바,X학과,2,80,90,70,17",          # 출석 범위
    "7,사,X학과,2, 80 , 90 ,-1,16",      # 과제 범위
    "8,아,X학과,3, 70 , 60 , 50 , 12 ",  # 정상 (공백)
]) + "\n", encoding="utf-8")
check("잘못된 행 6개 건너뜀", lambda: load_students(str(bad))[1] == 6)
check("정상 행 2개와 공백 처리", lambda: [(r["student_id"], r["midterm"], r["attendance"]) for r in load_students(str(bad))[0]] == [("1", 80, 16), ("8", 70, 12)])

---
## 문제 2. 성적 계산과 학과별 요약 (25점)

1. `total_score(record)`: 성적 규정에 따른 총점
2. `final_grade(record)`: 출석 규정을 반영한 최종 등급 `"A"`~`"F"`
3. `dept_summary(records)`: `{학과: {"count": 인원, "avg": 총점 평균(소수 첫째 자리), "pass_rate": 통과율}}`
   - **통과** = 최종 등급이 F가 아님
   - `pass_rate`는 백분율로 소수 첫째 자리 (예: 81.3)
   - 학과는 **가나다순**으로 들어 있어야 합니다 (`dict`는 넣은 순서를 기억합니다)

In [ ]:
def total_score(record):
    # TODO: 여기에 코드를 작성하세요
    raise NotImplementedError("코드를 작성하세요")


def final_grade(record):
    # TODO: 여기에 코드를 작성하세요
    raise NotImplementedError("코드를 작성하세요")


def dept_summary(records):
    # TODO: 여기에 코드를 작성하세요
    raise NotImplementedError("코드를 작성하세요")

In [ ]:
R = {"midterm": 80, "final": 90, "assignment": 85, "attendance": 15}
check("총점 85.5", lambda: total_score(R) == 85.5)
check("과제 None → 0점", lambda: total_score({**R, "assignment": None}) == 60.0)
check("등급 B", lambda: final_grade(R) == "B")
check("출석 11 → F", lambda: final_grade({**R, "midterm": 100, "final": 100, "assignment": 100, "attendance": 11}) == "F")
check("출석 12 경계 → A", lambda: final_grade({**R, "midterm": 100, "final": 100, "assignment": 100, "attendance": 12}) == "A")
check("총점 경계 90.0 → A", lambda: final_grade({"midterm": 90, "final": 90, "assignment": 90, "attendance": 16}) == "A")
check("총점 59.9 → F", lambda: final_grade({"midterm": 59, "final": 60, "assignment": 60.67, "attendance": 16}) == "F")

EXPECTED_CS = {"count": 17, "avg": 73.2, "pass_rate": 82.4}     # 자동 확인용 기준값 (students.csv)
try:
    recs = load_students(STUDENTS)[0]
except NotImplementedError:      # 문제 1을 아직 풀지 않았을 때
    recs = []
check("학과 가나다순", lambda: list(dept_summary(recs)) == ["간호학과", "경영학과", "전자공학과", "정보보안학과", "컴퓨터공학과"])
check("인원 합계 80", lambda: sum(v["count"] for v in dept_summary(recs).values()) == 80)
check("컴퓨터공학과 요약", lambda: dept_summary(recs)["컴퓨터공학과"] == EXPECTED_CS)

---
## 문제 3. `GradeBook` 클래스 (25점)

레코드 리스트를 받아 성적 정보를 조회하는 클래스입니다. **입력 리스트를 바꾸지 않습니다.**

| 메서드 | 동작 |
|---|---|
| `__init__(records)` | 레코드 보관 |
| `find(student_id)` | 해당 학번의 레코드(딕셔너리), 없으면 `None` |
| `top(n)` | 총점 상위 n명의 **이름** 리스트. 동점이면 **학번 오름차순** |
| `distribution()` | `{"A": 개수, "B": ..., "F": ...}` — **5개 키가 모두** 있어야 함 (0명이어도) |
| `at_risk()` | 최종 등급 F인 학생의 `(이름, 총점)` 리스트, **총점 오름차순** (같으면 학번순) |
| `__len__` | 학생 수 (`len(book)`) |

In [ ]:
class GradeBook:
    def __init__(self, records):
        raise NotImplementedError("코드를 작성하세요")

    def find(self, student_id):
        raise NotImplementedError("코드를 작성하세요")

    def top(self, n):
        raise NotImplementedError("코드를 작성하세요")

    def distribution(self):
        raise NotImplementedError("코드를 작성하세요")

    def at_risk(self):
        raise NotImplementedError("코드를 작성하세요")

    def __len__(self):
        raise NotImplementedError("코드를 작성하세요")

In [ ]:
EXPECTED_TOP3 = ["황서윤", "장예린", "한현우"]     # 자동 확인용 기준값 (students.csv)
try:
    book_recs = load_students(STUDENTS)[0]
except NotImplementedError:      # 문제 1을 아직 풀지 않았을 때
    book_recs = []
check("len(book) == 80", lambda: len(GradeBook(book_recs)) == 80)
check("find 있는 학번", lambda: GradeBook(book_recs).find("20261000")["name"] == "황서윤")
check("find 없는 학번 → None", lambda: GradeBook(book_recs).find("99999999") is None)
check("distribution 키 5개, 합 80", lambda: (list(GradeBook(book_recs).distribution()) == list("ABCDF")
      and sum(GradeBook(book_recs).distribution().values()) == 80))
check("top(3)", lambda: GradeBook(book_recs).top(3) == EXPECTED_TOP3)
check("at_risk 총점 오름차순", lambda: [s for _, s in GradeBook(book_recs).at_risk()] == sorted(s for _, s in GradeBook(book_recs).at_risk()))
check("at_risk 인원 = F 개수", lambda: len(GradeBook(book_recs).at_risk()) == GradeBook(book_recs).distribution()["F"])

mini = [{"student_id": "2", "name": "나", "department": "X", "year": 1, "midterm": 80, "final": 80, "assignment": 80, "attendance": 16},
        {"student_id": "1", "name": "가", "department": "X", "year": 1, "midterm": 80, "final": 80, "assignment": 80, "attendance": 16},
        {"student_id": "3", "name": "다", "department": "X", "year": 1, "midterm": 99, "final": 99, "assignment": 99, "attendance": 5}]
check("top 동점 → 학번순 (다는 출석 F여도 총점 1위)", lambda: GradeBook(mini).top(3) == ["다", "가", "나"])
check("빈 GradeBook distribution", lambda: GradeBook([]).distribution() == {"A": 0, "B": 0, "C": 0, "D": 0, "F": 0})
check("입력 리스트 불변", lambda: (lambda lst: (GradeBook(lst).top(1), len(lst))[1] == 3)(list(mini)))

---
## 문제 4. 보고서 파일 만들기 (20점)

`write_report(book, path)`는 `GradeBook`의 정보를 아래 형식의 텍스트 파일(UTF-8)로 저장하고, **저장한 줄 수**를 돌려줍니다.

```text
[성적 분석 보고서]
전체 인원: 80명
평균 총점: 72.5
등급 분포: A 5 / B 20 / C 25 / D 14 / F 16
학과별 평균: 간호학과 71.0 / 경영학과 70.2 / ...
상위 3명: 김가나, 이다라, 박마바
위험 학생: 16명 (최저 총점 35.0)
```

- 숫자는 예시이며, 실제 값은 여러분의 함수로 계산합니다. **숫자를 직접 써 넣으면 안 됩니다.**
- 평균 총점은 전체 학생 총점의 평균을 소수 첫째 자리로 반올림합니다.
- 학과별 평균은 `dept_summary`의 순서(가나다순)를 따릅니다.
- 위험 학생이 0명이면 마지막 줄은 `위험 학생: 0명`입니다.
- 학생이 0명인 `GradeBook`은 보고서를 만들지 않고 `ValueError`를 냅니다.

In [ ]:
def write_report(book, path):
    # TODO: 여기에 코드를 작성하세요
    raise NotImplementedError("코드를 작성하세요")

In [ ]:
report_path = TMP / "report.txt"


def _report_lines(records):
    write_report(GradeBook(records), report_path)
    return report_path.read_text(encoding="utf-8").splitlines()


check("7줄을 쓰고 7을 돌려줌", lambda: write_report(GradeBook(book_recs), report_path) == 7 and len(_report_lines(book_recs)) == 7)
check("제목과 인원", lambda: _report_lines(book_recs)[:2] == ["[성적 분석 보고서]", "전체 인원: 80명"])
check("등급 분포 줄", lambda: _report_lines(book_recs)[3] == "등급 분포: " + " / ".join(f"{g} {n}" for g, n in GradeBook(book_recs).distribution().items()))
check("상위 3명 줄", lambda: _report_lines(book_recs)[5] == "상위 3명: " + ", ".join(EXPECTED_TOP3))
check("위험 학생 0명 형식", lambda: _report_lines(mini[:2])[-1] == "위험 학생: 0명")

def _empty_raises():
    try:
        write_report(GradeBook([]), report_path)
    except ValueError:
        return True
    return False

check("빈 GradeBook → ValueError", _empty_raises)

In [ ]:
# 보고서 눈으로 확인하기
try:
    write_report(GradeBook(book_recs), report_path)
    print(report_path.read_text(encoding="utf-8"))
except (NotImplementedError, ValueError) as e:
    print("아직 보고서를 만들 수 없습니다:", type(e).__name__)

---
## 문제 5. 기술 메모 (10점)

아래 네 항목을 각각 2~4문장으로 작성하세요.

| 항목 | 쓸 내용 |
|---|---|
| 설계 | 함수·클래스를 어떻게 나누었고 왜 그렇게 했나 |
| 테스트 | 자동 확인 외에 직접 확인한 경계·오류 사례 |
| AI 사용 | 사용한 도구, 질문, 채택·수정한 부분 (사용하지 않았다면 "사용 안 함") |
| 한계 | 지금 코드가 처리하지 못하는 상황, 개선 방향 |

**설계**:

**테스트**:

**AI 사용**:

**한계**: